# 04 — MLOps Pipeline: Reentrenamiento Automatizado

**Proyecto:** Mantenimiento Predictivo Industrial  
**Modelos:** Autoencoder (detección de anomalías no supervisada) + Agente DQN (decisión de mantenimiento)  
**Dataset:** AI4I 2020 Predictive Maintenance Dataset (UCI)  

---

## Ciclo de Vida del Modelo en Producción

```
┌──────────────────────────────────────────────────────────────────┐
│                   CICLO DE VIDA MLOps                            │
│                                                                  │
│  Producción ──→ Monitoreo ──→ Detección Drift ──→ Trigger        │
│       ↑                                              │           │
│       │                                              ↓           │
│  Promoción  ←── Model Gate ←── Entrenamiento ←── Datos frescos  │
│       │                                                          │
│  Rollback (si gate falla)                                        │
└──────────────────────────────────────────────────────────────────┘
```

Este notebook documenta el **Paso 4** de la rúbrica:  
> *"Flujos de trabajo (pipelines) automatizados de mantenimiento e integración continua"*

## Sección 1 — Configuración del entorno MLOps

In [ ]:
import json
import sys
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

# Paths del proyecto
ROOT = Path('.').resolve()
API_DIR = ROOT.parent / 'api'
MODELS_DIR = API_DIR / 'models'
DATA_DIR = ROOT.parent / 'data'

sys.path.insert(0, str(API_DIR))

print(f'📁 ROOT:       {ROOT}')
print(f'📁 API_DIR:    {API_DIR}')
print(f'📁 MODELS_DIR: {MODELS_DIR}')
print(f'📁 DATA_DIR:   {DATA_DIR}')

# Leer metadata del modelo actual en producción
meta = json.loads((MODELS_DIR / 'model_metadata.json').read_text(encoding='utf-8'))
print(f'\n🤖 Modelo en producción: v{meta["version"]} — {meta["trained_at"]}')
print(f'   Threshold de anomalía: {meta["threshold"]:.6f}')
print(f'   MSE normal (línea base): {meta["metrics"]["reconstruction_mse_normal_mean"]:.6f}')

## Sección 2 — ¿Cuándo se dispara el reentrenamiento?

El sistema tiene **4 triggers** de reentrenamiento, todos integrados en GitHub Actions:

| Caso | Trigger | Workflow | Frecuencia |
|------|---------|----------|------------|
| **CASO 1** | Cron temporal | `retrain.yml` | Cada lunes 03:00 UTC |
| **CASO 2** | Data Drift | `drift_monitor.yml` | Cada hora (o manual) |
| **CASO 3** | Performance Degradation | `drift_monitor.yml` | Cada hora (o manual) |
| **CASO 4** | Manual | Cualquier workflow | Por demanda |


In [ ]:
# Visualizar los umbrales de drift y degradación
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

BASELINE_MSE = meta['metrics']['reconstruction_mse_normal_mean']
DRIFT_THRESHOLD = BASELINE_MSE * 1.30  # +30%
THRESHOLD_ANOMALY = meta['threshold']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Umbrales de Activación del Reentrenamiento', fontsize=14, fontweight='bold')

# --- CASO 2: Drift en MSE ---
ax1 = axes[0]
x = np.linspace(0, 100, 200)
mse_normal = np.random.normal(BASELINE_MSE, BASELINE_MSE * 0.1, 200)
mse_drift = np.concatenate([mse_normal[:130], np.random.normal(DRIFT_THRESHOLD * 1.1, 0.005, 70)])
ax1.plot(x, mse_drift, color='#2196F3', lw=1.5, label='MSE de reconstrucción (rolling mean)')
ax1.axhline(BASELINE_MSE, color='#4CAF50', ls='--', lw=2, label=f'Baseline: {BASELINE_MSE:.4f}')
ax1.axhline(DRIFT_THRESHOLD, color='#FF5722', ls='--', lw=2, label=f'Umbral drift (+30%): {DRIFT_THRESHOLD:.4f}')
ax1.axvspan(65, 100, alpha=0.15, color='red', label='Zona de drift')
ax1.set_title('CASO 2 — Drift de Datos')
ax1.set_xlabel('Tiempo (ventana de 200 predicciones)')
ax1.set_ylabel('MSE de Reconstrucción')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

# --- CASO 3: Alertas críticas ---
ax2 = axes[1]
hours = np.arange(24)
alerts_normal = np.array([np.random.randint(0, 3) for _ in range(20)] + [np.random.randint(5, 9) for _ in range(4)])
colors = ['#FF5722' if a >= 5 else '#4CAF50' for a in alerts_normal]
ax2.bar(hours, alerts_normal, color=colors, alpha=0.8)
ax2.axhline(5, color='#FF5722', ls='--', lw=2, label='Umbral crítico (=5 alertas/hora)')
ax2.set_title('CASO 3 — Degradación de Rendimiento')
ax2.set_xlabel('Hora del día')
ax2.set_ylabel('Alertas "critical" en la hora')
normal_patch = mpatches.Patch(color='#4CAF50', label='Normal')
crit_patch = mpatches.Patch(color='#FF5722', label='Activación reentrenamiento')
ax2.legend(handles=[normal_patch, crit_patch], fontsize=8)
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('drift_thresholds.png', dpi=120, bbox_inches='tight')
plt.show()
print('📊 Gráfico guardado como drift_thresholds.png')

## Sección 3 — Carga de datos frescos

En producción, esta función apuntaría a la base de datos o feature store.  
Para este demo usamos el dataset AI4I 2020 (disponible en `data/ai4i2020.csv` o generado sintéticamente).

In [ ]:
import retrain_script as rs

# Intentar cargar datos reales; si no existen, generar sintéticos
data_path = DATA_DIR / 'ai4i2020.csv'
if not data_path.exists():
    print('ℹ️  Dataset real no encontrado. Generando datos sintéticos...')
    data_path = rs._generate_synthetic_data()

df = pd.read_csv(data_path)
print(f'✅ Dataset cargado: {len(df)} filas × {len(df.columns)} columnas')
print(f'   Distribución Machine failure: {df["Machine failure"].value_counts().to_dict() if "Machine failure" in df.columns else "N/A"}')
print(f'\n   Estadísticas:')
SENSOR_COLS = ['Air temperature [K]', 'Process temperature [K]',
               'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
display(df[SENSOR_COLS].describe().round(3))

## Sección 4 — PASO 1: Backup del modelo actual

In [ ]:
# Backup antes de cualquier reentrenamiento
backup_dir = rs.backup_current_model()
print(f'\n✅ Backup creado en: {backup_dir}')
print('   Archivos respaldados:')
for f in backup_dir.iterdir():
    size_kb = f.stat().st_size / 1024
    print(f'   📄 {f.name}  ({size_kb:.1f} KB)')

## Sección 5 — PASO 2: Reentrenamiento del Autoencoder + DQN

In [ ]:
import time

OUTPUT_DIR = API_DIR / 'models_new'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('🔄 Iniciando reentrenamiento...')
print('   Esto puede tomar 1-3 minutos en la primera ejecución.\n')

t0 = time.time()
metrics, threshold = rs.run_training(data_path, OUTPUT_DIR)
elapsed = time.time() - t0

print(f'\n✅ Reentrenamiento completado en {elapsed:.1f}s')
print(f'\n📊 Métricas del modelo nuevo:')
for k, v in metrics.items():
    print(f'   {k}: {v}')
print(f'   Nuevo threshold: {threshold:.6f}')

## Sección 6 — PASO 3: Model Gate (Validación de métricas)

In [ ]:
print('🔍 Ejecutando Model Gate...')
print(f'   Criterios:')
print(f'   ├─ Mínimo de filas de entrenamiento: {rs.GATE_MIN_ROWS}')
print(f'   ├─ MSE nuevo ≤ MSE viejo × {rs.GATE_MAX_MSE_RATIO:.2f} (no más de {(rs.GATE_MAX_MSE_RATIO-1)*100:.0f}% peor)')
print(f'   └─ % fallas detectadas ≥ {rs.GATE_MIN_DETECTION_PCT}%\n')

passed = rs.validate_model(OUTPUT_DIR)

# Leer y mostrar el reporte
gate_report = json.loads(rs.GATE_REPORT_PATH.read_text(encoding='utf-8'))

status_icon = '✅' if passed else '❌'
print(f'\n{status_icon} Estado del gate: {"APROBADO" if passed else "RECHAZADO"}')
if gate_report['reject_reasons']:
    print(f'   Razones de rechazo: {gate_report["reject_reasons"]}')

In [ ]:
# Visualización comparativa: modelo viejo vs nuevo
fig, ax = plt.subplots(figsize=(10, 5))

labels = ['MSE Normal\n(↓ mejor)', 'Threshold\n(ajustado)', '% Fallas\nDetectadas\n(↑ mejor)']
old_values = [
    meta['metrics']['reconstruction_mse_normal_mean'],
    meta['threshold'],
    meta['metrics'].get('pct_fallas_sobre_umbral', 32.7),
]
new_values = [
    gate_report['new_mse'],
    threshold,
    metrics['pct_fallas_sobre_umbral'],
]

x = np.arange(len(labels))
width = 0.35
bars1 = ax.bar(x - width/2, old_values, width, label=f'Modelo Actual (v{meta["version"]})',
               color='#607D8B', alpha=0.8)
bars2 = ax.bar(x + width/2, new_values, width, label='Modelo Nuevo',
               color='#2196F3' if passed else '#F44336', alpha=0.8)

ax.set_title(f'Model Gate: Comparación Viejo vs Nuevo — {"✅ APROBADO" if passed else "❌ RECHAZADO"}')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.grid(alpha=0.3, axis='y')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() * 1.01,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() * 1.01,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('model_gate_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## Sección 7 — PASO 4: Promoción del modelo (o Rollback si falla el gate)

In [ ]:
if passed:
    print('✅ Gate aprobado — Promoviendo modelo a producción...')
    rs.promote_model(OUTPUT_DIR)
    
    # Verificar la actualización
    updated_meta = json.loads((MODELS_DIR / 'model_metadata.json').read_text(encoding='utf-8'))
    print(f'\n🚀 MODELO PROMOVIDO A PRODUCCIÓN')
    print(f'   Versión anterior: v{meta["version"]}  →  Nueva versión: v{updated_meta["version"]}')
    print(f'   Threshold:        {meta["threshold"]:.6f}  →  {updated_meta["threshold"]:.6f}')
    print(f'   Entrenado el:     {updated_meta["trained_at"]}')
    print(f'   Historial (últimas 3 entradas):')
    for h in updated_meta['retrain_history'][-3:]:
        print(f'     • v{h["version"]} | {h["trained_at"]} | {h["trigger"]} | {h["status"]}')
else:
    print('❌ Gate rechazado — El modelo nuevo NO llega a producción.')
    print('   En GitHub Actions el step de rollback se ejecuta automáticamente.')
    print('   Demostrando rollback manual:')
    rs.rollback()
    print('\n✅ Modelo anterior restaurado desde backup.')

## Sección 8 — GitHub Actions: Flujo de eventos

Todos los pasos anteriores se ejecutan automáticamente en GitHub Actions.  
A continuación se muestra el diagrama de los 3 workflows y cómo interactúan:

In [ ]:
diagram = """
╔══════════════════════════════════════════════════════════════════╗
║               GITHUB ACTIONS — 3 WORKFLOWS MLOps                ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  [ci-cd.yml]  → En cada push a main                             ║
║    job 1: test       → pytest test_api.py + test_retrain.py     ║
║    job 2: build      → docker build api/ + app/                 ║
║    job 3: deploy     → trigger redeploy en Render               ║
║                                                                  ║
║  [drift_monitor.yml]  → Cada hora (o manual para demo)          ║
║    job: check-drift   → python check_drift.py                   ║
║      ├─ CASO 2: drift_score > 30%  ─┐                           ║
║      └─ CASO 3: >5 critical/hora   ─┤→ job: trigger-retrain     ║
║                                      └→ dispara retrain.yml     ║
║                                                                  ║
║  [retrain.yml]  → Lunes 03:00 UTC / por drift / manual          ║
║    job: retrain                                                  ║
║      PASO 1: backup        → models_backup/v{ver}_{ts}/          ║
║      PASO 2: train         → python retrain_script.py           ║
║      PASO 3: gate          → validate_model()  ─[FAIL]→ exit 1  ║
║      PASO 4: promote       → models_new/ → api/models/          ║
║      PASO 5: rollback      → (solo si failure())                ║
║    job: post-retrain-tests → API health check + metadata check  ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(diagram)

## Sección 9 — Script ejecutable (resumen para el YAML)

El archivo `api/retrain_script.py` implementa todos los pasos anteriores como CLI:  

```bash
# Usado en retrain.yml:
python api/retrain_script.py --backup
python api/retrain_script.py --data-path data/ai4i2020.csv
python api/retrain_script.py --validate-only
python api/retrain_script.py --promote
python api/retrain_script.py --rollback    # solo si failure()

# Para demo local (no toca api/models/):
python api/retrain_script.py --dry-run
```

In [ ]:
# Estado final del sistema MLOps
final_meta = json.loads((MODELS_DIR / 'model_metadata.json').read_text(encoding='utf-8'))

print('=' * 60)
print('ESTADO FINAL DEL SISTEMA')
print('=' * 60)
print(f'  Modelo en producción: v{final_meta["version"]}')
print(f'  Threshold:            {final_meta["threshold"]:.6f}')
print(f'  Entrenado:            {final_meta["trained_at"]}')
print(f'  Reentrenamientos:     {len(final_meta["retrain_history"])}')
print(f'  Backup disponible:    {(rs.BACKUP_BASE_DIR / "latest.txt").exists()}')
print('=' * 60)
print('\n✅ Pipeline de reentrenamiento documentado y funcional.')